In [ ]:
import pandas as pd
from pathlib import Path

# Define file paths
raw_path = Path("data/raw")
processed_path = Path("data/processed")

# Create processed folder if it doesn't exist
processed_path.mkdir(parents=True, exist_ok=True)

# Load NAV history
nav_file = next(raw_path.glob("1788499983331-4389156d-02_nav_history.csv"))

nav_df = pd.read_csv(nav_file)

print("Raw NAV History Shape:", nav_df.shape)
print("\nColumns:")
print(nav_df.columns.tolist())

print("\nMissing Values:")
print(nav_df.isnull().sum())

print("\nDuplicate Rows:", nav_df.duplicated().sum())

In [2]:
from pathlib import Path

print("Jupyter is currently working in:")
print(Path.cwd())

print("\nFolders/files in the current folder:")
for item in Path.cwd().iterdir():
    print(item)

Jupyter is currently working in:
C:\Users\sarth\Downloads\capstone mf analysis project\notebooks

Folders/files in the current folder:
C:\Users\sarth\Downloads\capstone mf analysis project\notebooks\.ipynb_checkpoints
C:\Users\sarth\Downloads\capstone mf analysis project\notebooks\data
C:\Users\sarth\Downloads\capstone mf analysis project\notebooks\Day1_Data_Ingestion.ipynb
C:\Users\sarth\Downloads\capstone mf analysis project\notebooks\Day2_Data_Cleaning.ipynb
C:\Users\sarth\Downloads\capstone mf analysis project\notebooks\Day2_Data_Cleaning.ipynb.ipynb


In [3]:
from pathlib import Path
import pandas as pd

# Go from notebooks/ back to the project folder
project_path = Path("..")

raw_path = project_path / "data" / "raw"
processed_path = project_path / "data" / "processed"

# Create processed folder if it doesn't exist
processed_path.mkdir(parents=True, exist_ok=True)

# Find NAV history file
nav_file = next(raw_path.glob("*02_nav_history.csv"))

print("NAV file found:")
print(nav_file)

# Load the data
nav_df = pd.read_csv(nav_file)

print("\nRaw NAV History Shape:", nav_df.shape)

print("\nColumns:")
print(nav_df.columns.tolist())

print("\nMissing Values:")
print(nav_df.isnull().sum())

print("\nDuplicate Rows:", nav_df.duplicated().sum())

NAV file found:
..\data\raw\1788499983331-4389156d-02_nav_history.csv

Raw NAV History Shape: (46000, 3)

Columns:
['amfi_code', 'date', 'nav']

Missing Values:
amfi_code    0
date         0
nav          0
dtype: int64

Duplicate Rows: 0


In [6]:
# Convert date column to datetime
nav_df["date"] = pd.to_datetime(nav_df["date"], errors="coerce")

# Check if any dates failed to convert
print("Invalid dates:", nav_df["date"].isna().sum())

# Sort by AMFI code and date
nav_df = nav_df.sort_values(
    by=["amfi_code", "date"]
).reset_index(drop=True)

# Display the first 10 rows after sorting
print("\nFirst 10 rows after cleaning date and sorting:")
print(nav_df.head(10))

# Check data types
print("\nData types:")
print(nav_df.dtypes)

Invalid dates: 0

First 10 rows after cleaning date and sorting:
   amfi_code       date       nav
0     100016 2022-01-03  520.4608
1     100016 2022-01-04  515.0971
2     100016 2022-01-05  521.7239
3     100016 2022-01-06  515.7880
4     100016 2022-01-07  515.1639
5     100016 2022-01-10  510.7136
6     100016 2022-01-11  513.5542
7     100016 2022-01-12  512.3195
8     100016 2022-01-13  510.2445
9     100016 2022-01-14  514.3636

Data types:
amfi_code             int64
date         datetime64[ns]
nav                 float64
dtype: object


In [4]:
# Check duplicate AMFI code + date combinations
duplicate_keys = nav_df.duplicated(
    subset=["amfi_code", "date"],
    keep=False
)

print("Duplicate AMFI code + date records:", duplicate_keys.sum())

if duplicate_keys.sum() > 0:
    print("\nDuplicate records:")
    print(
        nav_df.loc[duplicate_keys]
        .sort_values(["amfi_code", "date"])
    )
else:
    print("No duplicate AMFI code + date combinations found.")

Duplicate AMFI code + date records: 0
No duplicate AMFI code + date combinations found.


In [5]:

invalid_nav = nav_df[nav_df["nav"] <= 0]

print("NAV values <= 0:", len(invalid_nav))

if len(invalid_nav) > 0:
    print("\nInvalid NAV records:")
    print(invalid_nav.head(20))
else:
    print("All NAV values are greater than 0.")

NAV values <= 0: 0
All NAV values are greater than 0.


In [6]:
print("Missing NAV values:", nav_df["nav"].isna().sum())

if nav_df["nav"].isna().sum() > 0:
    print("\nRecords with missing NAV:")
    print(nav_df[nav_df["nav"].isna()].head(20))
else:
    print("No missing NAV values found.")

Missing NAV values: 0
No missing NAV values found.


In [10]:
# Check the date range for each fund
date_summary = nav_df.groupby("amfi_code")["date"].agg(
    first_date="min",
    last_date="max",
    number_of_dates="count"
).reset_index()

print(date_summary.head(10))
print("\nNumber of funds:", len(date_summary))

   amfi_code first_date  last_date  number_of_dates
0     100016 2022-01-03 2026-05-29             1150
1     100025 2022-01-03 2026-05-29             1150
2     100033 2022-01-03 2026-05-29             1150
3     101206 2022-01-03 2026-05-29             1150
4     101207 2022-01-03 2026-05-29             1150
5     101208 2022-01-03 2026-05-29             1150
6     102885 2022-01-03 2026-05-29             1150
7     102886 2022-01-03 2026-05-29             1150
8     102887 2022-01-03 2026-05-29             1150
9     118632 2022-01-03 2026-05-29             1150

Number of funds: 40


In [11]:
# Check whether dates are consecutive within each fund
nav_df["date_diff"] = nav_df.groupby("amfi_code")["date"].diff().dt.days

print("Date gap distribution:")
print(nav_df["date_diff"].value_counts().sort_index())

Date gap distribution:
date_diff
1.0    36800
3.0     9160
Name: count, dtype: int64


In [12]:
# Remove temporary validation column
nav_df = nav_df.drop(columns=["date_diff"])

print("Current NAV columns:")
print(nav_df.columns.tolist())

print("\nFinal NAV shape:", nav_df.shape)

Current NAV columns:
['amfi_code', 'date', 'nav']

Final NAV shape: (46000, 3)


In [13]:
nav_output = processed_path / "nav_history_cleaned.csv"

nav_df.to_csv(nav_output, index=False)

print("Cleaned NAV file saved successfully.")
print("File:", nav_output)
print("Rows:", len(nav_df))

Cleaned NAV file saved successfully.
File: ..\data\processed\nav_history_cleaned.csv
Rows: 46000


In [7]:
# Load investor transactions
transactions_file = next(raw_path.glob("1788499980509-304c1255-08_invester_transactions.csv"))
transactions_df = pd.read_csv(transactions_file)

print("Transactions file found:")
print(transactions_file)

print("\nRaw Transactions Shape:", transactions_df.shape)

print("\nColumns:")
print(transactions_df.columns.tolist())

print("\nMissing Values:")
print(transactions_df.isnull().sum())

print("\nDuplicate Rows:", transactions_df.duplicated().sum())

StopIteration: 

In [17]:
# Find the investor transactions CSV automatically
transaction_files = list(raw_path.glob("*investor_transactions.csv"))

print("Matching files found:")
print(transaction_files)

Matching files found:
[WindowsPath('../data/raw/1788499980509-304c1255-08_investor_transactions.csv')]


In [19]:
transactions_file = transaction_files[0]
transactions_df = pd.read_csv(transactions_file)

print("Transactions file found:")
print(transactions_file)

print("\nRaw Transactions Shape:", transactions_df.shape)

print("\nColumns:")
print(transactions_df.columns.tolist())

print("\nMissing Values:")
print(transactions_df.isnull().sum())

print("\nDuplicate Rows:", transactions_df.duplicated().sum())

Transactions file found:
..\data\raw\1788499980509-304c1255-08_investor_transactions.csv

Raw Transactions Shape: (32778, 13)

Columns:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

Missing Values:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Duplicate Rows: 0


In [20]:
print("Transaction Type Values:")
print(transactions_df["transaction_type"].value_counts())

print("\nUnique Transaction Types:")
print(transactions_df["transaction_type"].unique())

Transaction Type Values:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

Unique Transaction Types:
['SIP' 'Redemption' 'Lumpsum']


In [21]:
invalid_amounts = transactions_df[
    transactions_df["amount_inr"] <= 0
]

print("Transactions with amount <= 0:", len(invalid_amounts))

if len(invalid_amounts) > 0:
    print("\nInvalid transactions:")
    print(invalid_amounts.head(20))
else:
    print("All transaction amounts are greater than 0.")

Transactions with amount <= 0: 0
All transaction amounts are greater than 0.


In [22]:
# Convert transaction date to datetime
transactions_df["transaction_date"] = pd.to_datetime(
    transactions_df["transaction_date"],
    errors="coerce"
)

print("Invalid transaction dates:",
      transactions_df["transaction_date"].isna().sum())

print("\nData type:")
print(transactions_df["transaction_date"].dtype)

print("\nDate range:")
print("First transaction:", transactions_df["transaction_date"].min())
print("Last transaction:", transactions_df["transaction_date"].max())

Invalid transaction dates: 0

Data type:
datetime64[ns]

Date range:
First transaction: 2024-01-01 00:00:00
Last transaction: 2025-05-30 00:00:00


In [23]:
print("KYC Status Values:")
print(transactions_df["kyc_status"].value_counts())

print("\nUnique KYC Status Values:")
print(transactions_df["kyc_status"].unique())

KYC Status Values:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Unique KYC Status Values:
['Verified' 'Pending']


In [24]:
print("Transaction Data Types:")
print(transactions_df.dtypes)

print("\nFinal Transactions Shape:")
print(transactions_df.shape)

Transaction Data Types:
investor_id                   object
transaction_date      datetime64[ns]
amfi_code                      int64
transaction_type              object
amount_inr                     int64
state                         object
city                          object
city_tier                     object
age_group                     object
gender                        object
annual_income_lakh           float64
payment_mode                  object
kyc_status                    object
dtype: object

Final Transactions Shape:
(32778, 13)


In [25]:
transactions_output = processed_path / "investor_transactions_cleaned.csv"

transactions_df.to_csv(transactions_output, index=False)

print("Cleaned Transactions file saved successfully.")
print("File:", transactions_output)
print("Rows:", len(transactions_df))
print("Columns:", len(transactions_df.columns))

Cleaned Transactions file saved successfully.
File: ..\data\processed\investor_transactions_cleaned.csv
Rows: 32778
Columns: 13


In [26]:
# Find scheme performance CSV
performance_files = list(raw_path.glob("*scheme_performance.csv"))

print("Matching files found:")
print(performance_files)

Matching files found:
[WindowsPath('../data/raw/1788499985420-bb134abf-07_scheme_performance.csv')]


In [27]:
performance_file = performance_files[0]
performance_df = pd.read_csv(performance_file)

print("Scheme Performance file found:")
print(performance_file)

print("\nRaw Shape:", performance_df.shape)

print("\nColumns:")
print(performance_df.columns.tolist())

print("\nMissing Values:")
print(performance_df.isnull().sum())

print("\nDuplicate Rows:", performance_df.duplicated().sum())

Scheme Performance file found:
..\data\raw\1788499985420-bb134abf-07_scheme_performance.csv

Raw Shape: (40, 19)

Columns:
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']

Missing Values:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

Duplicate Rows: 0


In [28]:
print("Scheme Performance Data Types:")
print(performance_df.dtypes)

Scheme Performance Data Types:
amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object


In [29]:
performance_columns = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct"
]

print("Performance metric ranges:")

for column in performance_columns:
    print(
        f"{column}: "
        f"Min = {performance_df[column].min()}, "
        f"Max = {performance_df[column].max()}"
    )

Performance metric ranges:
return_1yr_pct: Min = 4.26, Max = 24.93
return_3yr_pct: Min = 5.14, Max = 23.39
return_5yr_pct: Min = 5.43, Max = 23.8
benchmark_3yr_pct: Min = 3.96, Max = 22.16
alpha: Min = 0.51, Max = 1.98
beta: Min = 0.22, Max = 1.04
sharpe_ratio: Min = 0.8, Max = 7.68
sortino_ratio: Min = 1.03, Max = 10.37
std_dev_ann_pct: Min = 0.5, Max = 25.0
max_drawdown_pct: Min = -33.5, Max = -2.23


In [30]:
invalid_expense = performance_df[
    (performance_df["expense_ratio_pct"] < 0.1) |
    (performance_df["expense_ratio_pct"] > 2.5)
]

print("Expense ratios outside 0.1% - 2.5%:", len(invalid_expense))

if len(invalid_expense) > 0:
    print("\nInvalid expense ratio records:")
    print(
        invalid_expense[
            ["amfi_code", "scheme_name", "expense_ratio_pct"]
        ]
    )
else:
    print("All expense ratios are within the expected range.")

Expense ratios outside 0.1% - 2.5%: 0
All expense ratios are within the expected range.


In [31]:
print("Final Scheme Performance Shape:", performance_df.shape)

print("\nMissing values:")
print(performance_df.isnull().sum().sum())

print("\nDuplicate rows:")
print(performance_df.duplicated().sum())

Final Scheme Performance Shape: (40, 19)

Missing values:
0

Duplicate rows:
0


In [32]:
performance_output = processed_path / "scheme_performance_cleaned.csv"

performance_df.to_csv(performance_output, index=False)

print("Saved cleaned file:")
print(performance_output)
print("Rows saved:", len(performance_df))

Saved cleaned file:
..\data\processed\scheme_performance_cleaned.csv
Rows saved: 40


In [9]:
from pathlib import Path
import pandas as pd

# Project paths
project_path = Path("..")
raw_path = project_path / "data" / "raw"

# Load required datasets
fund_master_df = pd.read_csv(next(raw_path.glob("*01_fund_master.csv")))
nav_df = pd.read_csv(next(raw_path.glob("*02_nav_history.csv")))
transactions_df = pd.read_csv(next(raw_path.glob("*08_investor_transactions.csv")))
performance_df = pd.read_csv(next(raw_path.glob("*07_scheme_performance.csv")))

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [10]:
print("=== FUND MASTER ===")
print(fund_master_df.columns.tolist())

print("\n=== NAV HISTORY ===")
print(nav_df.columns.tolist())

print("\n=== INVESTOR TRANSACTIONS ===")
print(transactions_df.columns.tolist())

print("\n=== SCHEME PERFORMANCE ===")
print(performance_df.columns.tolist())

=== FUND MASTER ===
['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']

=== NAV HISTORY ===
['amfi_code', 'date', 'nav']

=== INVESTOR TRANSACTIONS ===
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

=== SCHEME PERFORMANCE ===
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


In [11]:
# Load AUM by fund house
aum_df = pd.read_csv(next(raw_path.glob("*03_aum_by_fund_house.csv")))

print("=== AUM BY FUND HOUSE ===")
print(aum_df.columns.tolist())

print("\nShape:")
print(aum_df.shape)

print("\nFirst 5 rows:")
display(aum_df.head())

print("\nData types:")
print(aum_df.dtypes)

=== AUM BY FUND HOUSE ===
['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']

Shape:
(90, 5)

First 5 rows:


,date,fund_house,aum_lakh_crore,aum_crore,num_schemes
0,2022-03-31,SBI Mutual Fund,6.05,605000,186
1,2022-03-31,ICICI Prudential MF,4.65,465000,216
2,2022-03-31,HDFC Mutual Fund,4.35,435000,195
3,2022-03-31,Nippon India MF,2.70,270000,177
4,2022-03-31,Kotak Mahindra MF,2.70,270000,168



Data types:
date               object
fund_house         object
aum_lakh_crore    float64
aum_crore           int64
num_schemes         int64
dtype: object


In [12]:
schema_sql = """
PRAGMA foreign_keys = ON;

-- ============================================
-- DIMENSION TABLE: FUND
-- ============================================

CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT NOT NULL,
    scheme_name TEXT NOT NULL,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date DATE,
    benchmark TEXT,
    expense_ratio_pct REAL,
    exit_load_pct REAL,
    min_sip_amount REAL,
    min_lumpsum_amount REAL,
    fund_manager TEXT,
    risk_category TEXT,
    sebi_category_code TEXT
);

-- ============================================
-- DIMENSION TABLE: DATE
-- ============================================

CREATE TABLE IF NOT EXISTS dim_date (
    date DATE PRIMARY KEY,
    year INTEGER,
    month INTEGER,
    month_name TEXT,
    quarter INTEGER,
    day INTEGER,
    day_of_week TEXT
);

-- ============================================
-- FACT TABLE: NAV
-- ============================================

CREATE TABLE IF NOT EXISTS fact_nav (
    amfi_code INTEGER NOT NULL,
    date DATE NOT NULL,
    nav REAL NOT NULL,

    PRIMARY KEY (amfi_code, date),

    FOREIGN KEY (amfi_code)
        REFERENCES dim_fund(amfi_code),

    FOREIGN KEY (date)
        REFERENCES dim_date(date)
);

-- ============================================
-- FACT TABLE: TRANSACTIONS
-- ============================================

CREATE TABLE IF NOT EXISTS fact_transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id TEXT NOT NULL,
    transaction_date DATE NOT NULL,
    amfi_code INTEGER NOT NULL,
    transaction_type TEXT NOT NULL,
    amount_inr REAL NOT NULL,
    state TEXT,
    city TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    annual_income_lakh REAL,
    payment_mode TEXT,
    kyc_status TEXT,

    FOREIGN KEY (amfi_code)
        REFERENCES dim_fund(amfi_code),

    FOREIGN KEY (transaction_date)
        REFERENCES dim_date(date)
);

-- ============================================
-- FACT TABLE: PERFORMANCE
-- ============================================

CREATE TABLE IF NOT EXISTS fact_performance (
    amfi_code INTEGER PRIMARY KEY,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    benchmark_3yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    aum_crore REAL,
    expense_ratio_pct REAL,
    morningstar_rating INTEGER,
    risk_grade TEXT,

    FOREIGN KEY (amfi_code)
        REFERENCES dim_fund(amfi_code)
);

-- ============================================
-- FACT TABLE: AUM
-- ============================================

CREATE TABLE IF NOT EXISTS fact_aum (
    date DATE NOT NULL,
    fund_house TEXT NOT NULL,
    aum_lakh_crore REAL,
    aum_crore REAL,
    num_schemes INTEGER,

    PRIMARY KEY (date, fund_house),

    FOREIGN KEY (date)
        REFERENCES dim_date(date)
);
"""

schema_path = project_path / "sql" / "schema.sql"

with open(schema_path, "w", encoding="utf-8") as f:
    f.write(schema_sql)

print("schema.sql created successfully!")
print(schema_path)

schema.sql created successfully!
..\sql\schema.sql


In [13]:
# Check processed files

processed_files = list(processed_path.glob("*.csv"))

print("Processed CSV files:")
for file in processed_files:
    print("-", file.name)

print("\nTotal processed CSV files:", len(processed_files))

Processed CSV files:
- investor_transactions_cleaned.csv
- nav_history_cleaned.csv
- scheme_performance_cleaned.csv

Total processed CSV files: 3


In [14]:
fund_master_df = pd.read_csv(
    next(raw_path.glob("*01_fund_master.csv"))
)

print("Fund Master shape:", fund_master_df.shape)
print("Duplicate rows:", fund_master_df.duplicated().sum())
print("Missing values:", fund_master_df.isnull().sum().sum())
print("Unique AMFI codes:", fund_master_df["amfi_code"].nunique())


Fund Master shape: (40, 15)
Duplicate rows: 0
Missing values: 0
Unique AMFI codes: 40


In [15]:
fund_master_df["launch_date"] = pd.to_datetime(
    fund_master_df["launch_date"],
    errors="coerce"
)

print("Invalid launch dates:",
      fund_master_df["launch_date"].isna().sum())

print("\nLaunch date data type:")
print(fund_master_df["launch_date"].dtype)

print("\nFinal dim_fund shape:")
print(fund_master_df.shape)

Invalid launch dates: 0

Launch date data type:
datetime64[ns]

Final dim_fund shape:
(40, 15)


In [16]:
# Prepare date columns

nav_df["date"] = pd.to_datetime(nav_df["date"], errors="coerce")
transactions_df["transaction_date"] = pd.to_datetime(
    transactions_df["transaction_date"],
    errors="coerce"
)
aum_df["date"] = pd.to_datetime(aum_df["date"], errors="coerce")

# Collect all dates used by the fact tables
all_dates = pd.concat([
    nav_df["date"],
    transactions_df["transaction_date"],
    aum_df["date"]
]).dropna().drop_duplicates().sort_values()

# Create date dimension
dim_date_df = pd.DataFrame({
    "date": all_dates
})

dim_date_df["year"] = dim_date_df["date"].dt.year
dim_date_df["month"] = dim_date_df["date"].dt.month
dim_date_df["month_name"] = dim_date_df["date"].dt.month_name()
dim_date_df["quarter"] = dim_date_df["date"].dt.quarter
dim_date_df["day"] = dim_date_df["date"].dt.day
dim_date_df["day_of_week"] = dim_date_df["date"].dt.day_name()

# Convert date to string format for SQLite
dim_date_df["date"] = dim_date_df["date"].dt.strftime("%Y-%m-%d")

print("dim_date shape:", dim_date_df.shape)

print("\nFirst 5 rows:")
display(dim_date_df.head())

print("\nLast 5 rows:")
display(dim_date_df.tail())

print("\nDuplicate dates:", dim_date_df["date"].duplicated().sum())

dim_date shape: (1297, 7)

First 5 rows:


,date,year,month,month_name,quarter,day,day_of_week
0,2022-01-03,2022,1,January,1,3,Monday
1,2022-01-04,2022,1,January,1,4,Tuesday
2,2022-01-05,2022,1,January,1,5,Wednesday
3,2022-01-06,2022,1,January,1,6,Thursday
4,2022-01-07,2022,1,January,1,7,Friday



Last 5 rows:


,date,year,month,month_name,quarter,day,day_of_week
1145,2026-05-25,2026,5,May,2,25,Monday
1146,2026-05-26,2026,5,May,2,26,Tuesday
1147,2026-05-27,2026,5,May,2,27,Wednesday
1148,2026-05-28,2026,5,May,2,28,Thursday
1149,2026-05-29,2026,5,May,2,29,Friday



Duplicate dates: 0


In [17]:
nav_cleaned_path = processed_path / "nav_history_cleaned.csv"

nav_cleaned_df = pd.read_csv(nav_cleaned_path)

nav_cleaned_df["date"] = pd.to_datetime(
    nav_cleaned_df["date"],
    errors="coerce"
)

print("fact_nav shape:", nav_cleaned_df.shape)

print("\nColumns:")
print(nav_cleaned_df.columns.tolist())

print("\nMissing values:")
print(nav_cleaned_df.isnull().sum())

print("\nDuplicate (amfi_code, date) pairs:",
      nav_cleaned_df.duplicated(
          subset=["amfi_code", "date"]
      ).sum())

print("\nNAV <= 0:",
      (nav_cleaned_df["nav"] <= 0).sum())

fact_nav shape: (46000, 3)

Columns:
['amfi_code', 'date', 'nav']

Missing values:
amfi_code    0
date         0
nav          0
dtype: int64

Duplicate (amfi_code, date) pairs: 0

NAV <= 0: 0


In [18]:
transactions_cleaned_path = processed_path / "investor_transactions_cleaned.csv"

transactions_cleaned_df = pd.read_csv(transactions_cleaned_path)

transactions_cleaned_df["transaction_date"] = pd.to_datetime(
    transactions_cleaned_df["transaction_date"],
    errors="coerce"
)

print("fact_transactions shape:", transactions_cleaned_df.shape)

print("\nColumns:")
print(transactions_cleaned_df.columns.tolist())

print("\nMissing values:")
print(transactions_cleaned_df.isnull().sum())

print("\nDuplicate rows:",
      transactions_cleaned_df.duplicated().sum())

print("\nAmount <= 0:",
      (transactions_cleaned_df["amount_inr"] <= 0).sum())

print("\nTransaction types:")
print(transactions_cleaned_df["transaction_type"].value_counts())

print("\nKYC status:")
print(transactions_cleaned_df["kyc_status"].value_counts())

fact_transactions shape: (32778, 13)

Columns:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

Missing values:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Duplicate rows: 0

Amount <= 0: 0

Transaction types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

KYC status:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


In [19]:

performance_cleaned_path = processed_path / "scheme_performance_cleaned.csv"

performance_cleaned_df = pd.read_csv(performance_cleaned_path)

print("fact_performance shape:", performance_cleaned_df.shape)

print("\nColumns:")
print(performance_cleaned_df.columns.tolist())

print("\nMissing values:",
      performance_cleaned_df.isnull().sum().sum())

print("\nDuplicate rows:",
      performance_cleaned_df.duplicated().sum())

print("\nUnique AMFI codes:",
      performance_cleaned_df["amfi_code"].nunique())

print("\nExpense ratio outside 0.1%-2.5%:",
      (
          (performance_cleaned_df["expense_ratio_pct"] < 0.1) |
          (performance_cleaned_df["expense_ratio_pct"] > 2.5)
      ).sum())





fact_performance shape: (40, 19)

Columns:
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']

Missing values: 0

Duplicate rows: 0

Unique AMFI codes: 40

Expense ratio outside 0.1%-2.5%: 0


In [20]:
# Prepare fact_aum

aum_fact_df = pd.read_csv(
    next(raw_path.glob("*03_aum_by_fund_house.csv"))
)

aum_fact_df["date"] = pd.to_datetime(
    aum_fact_df["date"],
    errors="coerce"
)

print("fact_aum shape:", aum_fact_df.shape)

print("\nColumns:")
print(aum_fact_df.columns.tolist())

print("\nMissing values:")
print(aum_fact_df.isnull().sum())

print("\nDuplicate (date, fund_house) pairs:",
      aum_fact_df.duplicated(
          subset=["date", "fund_house"]
      ).sum())

print("\nAUM <= 0:")
print((aum_fact_df["aum_crore"] <= 0).sum())

print("\nNumber of schemes <= 0:")
print((aum_fact_df["num_schemes"] <= 0).sum())

fact_aum shape: (90, 5)

Columns:
['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']

Missing values:
date              0
fund_house        0
aum_lakh_crore    0
aum_crore         0
num_schemes       0
dtype: int64

Duplicate (date, fund_house) pairs: 0

AUM <= 0:
0

Number of schemes <= 0:
0


In [21]:
from sqlalchemy import create_engine, text

# SQLite database path
db_path = project_path / "sql" / "bluestock_mf.db"

# Create SQLite engine
engine = create_engine(f"sqlite:///{db_path}")

# Read our schema.sql file
schema_path = project_path / "sql" / "schema.sql"

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

# Execute each CREATE TABLE statement
with engine.begin() as connection:
    statements = schema_sql.split(";")

    for statement in statements:
        statement = statement.strip()

        if statement:
            connection.execute(text(statement))

print("SQLite database created successfully!")
print("Database:", db_path)

SQLite database created successfully!
Database: ..\sql\bluestock_mf.db


In [22]:
# Prepare dim_fund for SQLite
dim_fund_load = fund_master_df.copy()

# Convert dates to SQLite-compatible YYYY-MM-DD format
dim_fund_load["launch_date"] = (
    pd.to_datetime(dim_fund_load["launch_date"], errors="coerce")
    .dt.strftime("%Y-%m-%d")
)

# Prepare dim_date
dim_date_load = dim_date_df.copy()

dim_date_load["date"] = (
    pd.to_datetime(dim_date_load["date"], errors="coerce")
    .dt.strftime("%Y-%m-%d")
)

# Load both dimension tables
dim_fund_load.to_sql(
    "dim_fund",
    engine,
    if_exists="append",
    index=False
)

dim_date_load.to_sql(
    "dim_date",
    engine,
    if_exists="append",
    index=False
)

print("dim_fund loaded:", len(dim_fund_load), "rows")
print("dim_date loaded:", len(dim_date_load), "rows")

dim_fund loaded: 40 rows
dim_date loaded: 1297 rows


In [24]:
# -----------------------------
# 1. fact_nav
# -----------------------------
fact_nav_load = pd.read_csv(
    processed_path / "nav_history_cleaned.csv"
)

fact_nav_load["date"] = (
    pd.to_datetime(
        fact_nav_load["date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")
)


# -----------------------------
# 2. fact_transactions
# -----------------------------
fact_transactions_load = pd.read_csv(
    processed_path / "investor_transactions_cleaned.csv"
)

fact_transactions_load["transaction_date"] = (
    pd.to_datetime(
        fact_transactions_load["transaction_date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")
)


# -----------------------------
# 3. fact_performance
# -----------------------------
fact_performance_load = pd.read_csv(
    processed_path / "scheme_performance_cleaned.csv"
)


# -----------------------------
# 4. fact_aum
# -----------------------------
fact_aum_load = pd.read_csv(
    next(raw_path.glob("*03_aum_by_fund_house.csv"))
)

fact_aum_load["date"] = (
    pd.to_datetime(
        fact_aum_load["date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")
)


# -----------------------------
# Check shapes
# -----------------------------
print("fact_nav:", fact_nav_load.shape)
print("fact_transactions:", fact_transactions_load.shape)
print("fact_performance:", fact_performance_load.shape)
print("fact_aum:", fact_aum_load.shape)

fact_nav: (46000, 3)
fact_transactions: (32778, 13)
fact_performance: (40, 19)
fact_aum: (90, 5)


In [25]:
# AMFI codes available in dim_fund
fund_codes = set(
    dim_fund_load["amfi_code"].astype(int)
)

# Dates available in dim_date
date_values = set(
    dim_date_load["date"]
)

# -----------------------------
# AMFI code checks
# -----------------------------

nav_orphans = (
    ~fact_nav_load["amfi_code"].astype(int).isin(fund_codes)
).sum()

transaction_orphans = (
    ~fact_transactions_load["amfi_code"].astype(int).isin(fund_codes)
).sum()

performance_orphans = (
    ~fact_performance_load["amfi_code"].astype(int).isin(fund_codes)
).sum()


# -----------------------------
# Date checks
# -----------------------------

nav_date_orphans = (
    ~fact_nav_load["date"].isin(date_values)
).sum()

transaction_date_orphans = (
    ~fact_transactions_load["transaction_date"].isin(date_values)
).sum()

aum_date_orphans = (
    ~fact_aum_load["date"].isin(date_values)
).sum()


# -----------------------------
# Results
# -----------------------------

print("NAV AMFI code orphans:", nav_orphans)
print("Transaction AMFI code orphans:", transaction_orphans)
print("Performance AMFI code orphans:", performance_orphans)

print("\nNAV date orphans:", nav_date_orphans)
print("Transaction date orphans:", transaction_date_orphans)
print("AUM date orphans:", aum_date_orphans)

NAV AMFI code orphans: 0
Transaction AMFI code orphans: 0
Performance AMFI code orphans: 0

NAV date orphans: 0
Transaction date orphans: 0
AUM date orphans: 0


In [27]:
# Load fact_nav
fact_nav_load.to_sql(
    "fact_nav",
    engine,
    if_exists="append",
    index=False
)

# Load fact_transactions
fact_transactions_load.to_sql(
    "fact_transactions",
    engine,
    if_exists="append",
    index=False
)

# Load fact_performance
fact_performance_load.to_sql(
    "fact_performance",
    engine,
    if_exists="append",
    index=False
)

# Load fact_aum
fact_aum_load.to_sql(
    "fact_aum",
    engine,
    if_exists="append",
    index=False
)

print("All fact tables loaded successfully!")

IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: fact_nav.amfi_code, fact_nav.date
[SQL: INSERT INTO fact_nav (amfi_code, date, nav) VALUES (?, ?, ?)]
[parameters: [(100016, '2022-01-03', 520.4608), (100016, '2022-01-04', 515.0971), (100016, '2022-01-05', 521.7239), (100016, '2022-01-06', 515.788), (100016, '2022-01-07', 515.1639), (100016, '2022-01-10', 510.7136), (100016, '2022-01-11', 513.5542), (100016, '2022-01-12', 512.3195)  ... displaying 10 of 46000 total bound parameter sets ...  (149324, '2026-05-28', 280.6873), (149324, '2026-05-29', 279.7511)]]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [28]:
from sqlalchemy import text

with engine.connect() as connection:
    for table in [
        "dim_fund",
        "dim_date",
        "fact_nav",
        "fact_transactions",
        "fact_performance",
        "fact_aum"
    ]:
        count = connection.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        ).scalar()

        print(f"{table}: {count}")

dim_fund: 40
dim_date: 1297
fact_nav: 46000
fact_transactions: 32778
fact_performance: 0
fact_aum: 0


In [29]:
# Load fact_performance
fact_performance_load.to_sql(
    "fact_performance",
    engine,
    if_exists="append",
    index=False
)

# Load fact_aum
fact_aum_load.to_sql(
    "fact_aum",
    engine,
    if_exists="append",
    index=False
)

print("fact_performance loaded:", len(fact_performance_load))
print("fact_aum loaded:", len(fact_aum_load))

OperationalError: (sqlite3.OperationalError) table fact_performance has no column named scheme_name
[SQL: INSERT INTO fact_performance (amfi_code, scheme_name, fund_house, category, "plan", return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: [(119551, 'SBI Bluechip Fund - Regular Plan - Growth', 'SBI Mutual Fund', 'Large Cap', 'Regular', 12.42, 12.36, 14.45, 11.49, 0.87, 0.89, 0.88, 1.29, 14.0, -21.7, 14288, 1.54, 4, 'Moderate'), (119552, 'SBI Bluechip Fund - Direct Plan - Growth', 'SBI Mutual Fund', 'Large Cap', 'Direct', 15.25, 11.3, 14.23, 9.52, 1.78, 0.87, 0.81, 1.29, 14.0, -24.43, 1231, 0.66, 3, 'Moderate'), (119598, 'SBI Small Cap Fund - Regular Plan - Growth', 'SBI Mutual Fund', 'Small Cap', 'Regular', 24.56, 23.39, 20.67, 22.16, 1.23, 0.89, 0.94, 1.

In [30]:
# Keep only columns defined in the fact_performance table

fact_performance_load = fact_performance_load[
    [
        "amfi_code",
        "return_1yr_pct",
        "return_3yr_pct",
        "return_5yr_pct",
        "benchmark_3yr_pct",
        "alpha",
        "beta",
        "sharpe_ratio",
        "sortino_ratio",
        "std_dev_ann_pct",
        "max_drawdown_pct",
        "aum_crore",
        "expense_ratio_pct",
        "morningstar_rating",
        "risk_grade"
    ]
]

print("fact_performance shape:", fact_performance_load.shape)
print("\nColumns:")
print(fact_performance_load.columns.tolist())

fact_performance shape: (40, 15)

Columns:
['amfi_code', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


In [31]:
# Load fact_performance
fact_performance_load.to_sql(
    "fact_performance",
    engine,
    if_exists="append",
    index=False
)

# Load fact_aum
fact_aum_load.to_sql(
    "fact_aum",
    engine,
    if_exists="append",
    index=False
)

print("fact_performance loaded:", len(fact_performance_load))
print("fact_aum loaded:", len(fact_aum_load))

fact_performance loaded: 40
fact_aum loaded: 90


In [32]:
# Verify row counts in SQLite

with engine.connect() as connection:
    tables = [
        "dim_fund",
        "dim_date",
        "fact_nav",
        "fact_transactions",
        "fact_performance",
        "fact_aum"
    ]

    for table in tables:
        result = connection.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        )
        count = result.scalar()
        print(f"{table}: {count}")

dim_fund: 40
dim_date: 1297
fact_nav: 46000
fact_transactions: 32778
fact_performance: 40
fact_aum: 90


In [33]:
query_1 = """
SELECT
    f.scheme_name,
    f.fund_house,
    p.aum_crore
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
ORDER BY p.aum_crore DESC
LIMIT 5;
"""

with engine.connect() as connection:
    result = connection.execute(text(query_1))

    for row in result:
        print(row)

('Mirae Asset Emerging Bluechip Fund - Regular - Growth', 'Mirae Asset MF', 49046.0)
('Kotak Emerging Equity Fund - Regular - Growth', 'Kotak Mahindra MF', 47469.0)
('Nippon India Small Cap Fund - Regular - Growth', 'Nippon India MF', 43630.0)
('DSP Top 100 Equity Fund - Regular - Growth', 'DSP Mutual Fund', 41828.0)
('UTI Mid Cap Fund - Regular - Growth', 'UTI Mutual Fund', 41728.0)


In [34]:
query_2 = """
SELECT
    strftime('%Y', date) AS year,
    strftime('%m', date) AS month,
    ROUND(AVG(nav), 2) AS average_nav
FROM fact_nav
GROUP BY year, month
ORDER BY year, month;
"""

with engine.connect() as connection:
    result = connection.execute(text(query_2))

    for row in result:
        print(row)

('2022', '01', 207.06)
('2022', '02', 207.72)
('2022', '03', 209.69)
('2022', '04', 211.83)
('2022', '05', 212.73)
('2022', '06', 213.86)
('2022', '07', 213.96)
('2022', '08', 215.68)
('2022', '09', 218.49)
('2022', '10', 219.53)
('2022', '11', 223.47)
('2022', '12', 226.76)
('2023', '01', 230.67)
('2023', '02', 233.85)
('2023', '03', 238.01)
('2023', '04', 240.61)
('2023', '05', 241.89)
('2023', '06', 244.61)
('2023', '07', 245.78)
('2023', '08', 247.73)
('2023', '09', 251.83)
('2023', '10', 255.61)
('2023', '11', 258.35)
('2023', '12', 262.02)
('2024', '01', 264.62)
('2024', '02', 266.33)
('2024', '03', 269.6)
('2024', '04', 271.3)
('2024', '05', 273.45)
('2024', '06', 275.47)
('2024', '07', 279.02)
('2024', '08', 279.74)
('2024', '09', 281.02)
('2024', '10', 282.13)
('2024', '11', 285.24)
('2024', '12', 285.27)
('2025', '01', 288.04)
('2025', '02', 291.74)
('2025', '03', 296.16)
('2025', '04', 299.41)
('2025', '05', 303.94)
('2025', '06', 306.88)
('2025', '07', 309.73)
('2025', '08'

In [35]:
query_3 = """
WITH yearly_sip AS (
    SELECT
        strftime('%Y', transaction_date) AS year,
        SUM(amount_inr) AS sip_amount
    FROM fact_transactions
    WHERE transaction_type = 'SIP'
    GROUP BY year
)

SELECT
    year,
    ROUND(sip_amount, 2) AS sip_amount,
    ROUND(
        (sip_amount - LAG(sip_amount) OVER (ORDER BY year))
        * 100.0
        / LAG(sip_amount) OVER (ORDER BY year),
        2
    ) AS yoy_growth_pct
FROM yearly_sip
ORDER BY year;
"""

with engine.connect() as connection:
    result = connection.execute(text(query_3))

    for row in result:
        print(row)

('2024', 153233052.0, None)
('2025', 64000439.0, -58.23)


In [36]:


query_4 = """
SELECT
    state,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount_inr), 2) AS total_transaction_amount
FROM fact_transactions
GROUP BY state
ORDER BY total_transaction_amount DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query_4))

    for row in result:
        print(row)

('Punjab', 2965, 315780459.0)
('Tamil Nadu', 2806, 315177237.0)
('Madhya Pradesh', 2931, 308312493.0)
('Rajasthan', 2577, 298645822.0)
('Gujarat', 2780, 298358940.0)
('West Bengal', 2748, 297182514.0)
('Telangana', 2718, 290219284.0)
('Delhi', 2677, 289633404.0)
('Uttar Pradesh', 2695, 285368873.0)
('Haryana', 2736, 279634354.0)
('Karnataka', 2621, 273753570.0)
('Maharashtra', 2524, 269513480.0)


In [37]:
query_5 = """
SELECT
    f.scheme_name,
    f.fund_house,
    p.expense_ratio_pct
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
WHERE p.expense_ratio_pct < 1
ORDER BY p.expense_ratio_pct ASC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query_5))

    for row in result:
        print(row)

('Nippon India Gilt Securities Fund - Regular - Growth', 'Nippon India MF', 0.55)
('HDFC Short Term Debt Fund - Regular - Growth', 'HDFC Mutual Fund', 0.56)
('Kotak Liquid Fund - Regular - Growth', 'Kotak Mahindra MF', 0.6)
('SBI Bluechip Fund - Direct Plan - Growth', 'SBI Mutual Fund', 0.66)
('Nippon India Large Cap Fund - Direct - Growth', 'Nippon India MF', 0.72)
('SBI Small Cap Fund - Direct Plan - Growth', 'SBI Mutual Fund', 0.72)
('ICICI Pru Liquid Fund - Regular - Growth', 'ICICI Prudential MF', 0.74)
('Axis Bluechip Fund - Direct - Growth', 'Axis Mutual Fund', 0.75)
('SBI Magnum Gilt Fund - Regular Plan - Growth', 'SBI Mutual Fund', 0.77)
('HDFC Mid-Cap Opportunities Fund - Direct - Growth', 'HDFC Mutual Fund', 0.78)
('ABSL Liquid Fund - Regular - Growth', 'Aditya Birla Sun Life MF', 0.79)
('ICICI Pru Bluechip Fund - Direct - Growth', 'ICICI Prudential MF', 0.8)
('Nippon India ETF Nifty 50 BeES', 'Nippon India MF', 0.89)
('HDFC Top 100 Fund - Direct Plan - Growth', 'HDFC Mutual

In [38]:
query_6 = """
SELECT
    transaction_type,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount_inr), 2) AS total_amount,
    ROUND(AVG(amount_inr), 2) AS average_amount
FROM fact_transactions
GROUP BY transaction_type
ORDER BY total_amount DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query_6))

    for row in result:
        print(row)

('Lumpsum', 8095, 2059821448.0, 254456.02)
('Redemption', 4967, 1244525491.0, 250558.79)
('SIP', 19716, 217233491.0, 11018.13)


In [39]:
query_7 = """
SELECT
    f.scheme_name,
    f.fund_house,
    p.return_3yr_pct,
    p.benchmark_3yr_pct,
    ROUND(
        p.return_3yr_pct - p.benchmark_3yr_pct,
        2
    ) AS performance_difference
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
WHERE p.return_3yr_pct > p.benchmark_3yr_pct
ORDER BY performance_difference DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query_7))

    for row in result:
        print(row)

('HDFC Short Term Debt Fund - Regular - Growth', 'HDFC Mutual Fund', 7.37, 5.39, 1.98)
('Kotak Emerging Equity Fund - Regular - Growth', 'Kotak Mahindra MF', 18.23, 16.32, 1.91)
('ICICI Pru Liquid Fund - Regular - Growth', 'ICICI Prudential MF', 7.68, 5.83, 1.85)
('Kotak Flexicap Fund - Regular - Growth', 'Kotak Mahindra MF', 15.65, 13.8, 1.85)
('ABSL Small Cap Fund - Regular - Growth', 'Aditya Birla Sun Life MF', 22.38, 20.54, 1.84)
('DSP Top 100 Equity Fund - Regular - Growth', 'DSP Mutual Fund', 12.82, 11.0, 1.82)
('Nippon India ETF Nifty 50 BeES', 'Nippon India MF', 11.77, 9.97, 1.8)
('UTI Flexi Cap Fund - Regular - Growth', 'UTI Mutual Fund', 15.34, 13.55, 1.79)
('SBI Bluechip Fund - Direct Plan - Growth', 'SBI Mutual Fund', 11.3, 9.52, 1.78)
('Nippon India Large Cap Fund - Direct - Growth', 'Nippon India MF', 12.33, 10.63, 1.7)
('Mirae Asset Emerging Bluechip Fund - Regular - Growth', 'Mirae Asset MF', 14.56, 12.86, 1.7)
('Mirae Asset Large Cap Fund - Regular - Growth', 'Mirae As

In [40]:

query_8 = """
SELECT
    city_tier,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount_inr), 2) AS total_transaction_amount,
    ROUND(AVG(amount_inr), 2) AS average_transaction_amount
FROM fact_transactions
GROUP BY city_tier
ORDER BY average_transaction_amount DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query_8))

    for row in result:
        print(row)


('B30', 11059, 1202325640.0, 108719.2)
('T30', 21719, 2319254790.0, 106784.6)


In [41]:
query_9 = """
SELECT
    risk_grade,
    COUNT(*) AS fund_count,
    ROUND(AVG(return_1yr_pct), 2) AS average_1yr_return,
    ROUND(AVG(return_3yr_pct), 2) AS average_3yr_return
FROM fact_performance
GROUP BY risk_grade
ORDER BY fund_count DESC;
"""

with engine.connect() as connection:
    result = connection.execute(text(query_9))

    for row in result:
        print(row)

('Moderate', 16, 13.5, 12.85)
('High', 8, 15.38, 16.21)
('Very High', 6, 22.26, 21.69)
('Low', 6, 6.28, 6.29)
('Moderately High', 4, 16.19, 15.08)


In [42]:


query_10 = """
SELECT
    f.scheme_name,
    f.fund_house,
    p.return_1yr_pct,
    p.return_3yr_pct,
    p.risk_grade
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
ORDER BY p.return_1yr_pct DESC
LIMIT 5;
"""

with engine.connect() as connection:
    result = connection.execute(text(query_10))

    for row in result:
        print(row)

('ABSL Small Cap Fund - Regular - Growth', 'Aditya Birla Sun Life MF', 24.93, 22.38, 'Very High')
('SBI Small Cap Fund - Regular Plan - Growth', 'SBI Mutual Fund', 24.56, 23.39, 'Very High')
('Axis Small Cap Fund - Regular - Growth', 'Axis Mutual Fund', 21.97, 20.98, 'Very High')
('Nippon India Small Cap Fund - Regular - Growth', 'Nippon India MF', 21.3, 20.15, 'Very High')
('SBI Small Cap Fund - Direct Plan - Growth', 'SBI Mutual Fund', 20.59, 23.14, 'Very High')


In [47]:
queries_path = project_path / "sql" / "queries.sql"

queries_sql = """
-- Query 1: Top 5 Funds by AUM
SELECT
    f.scheme_name,
    f.fund_house,
    p.aum_crore
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
ORDER BY p.aum_crore DESC
LIMIT 5;


-- Query 2: Average NAV per Month
SELECT
    strftime('%Y', date) AS year,
    strftime('%m', date) AS month,
    ROUND(AVG(nav), 2) AS average_nav
FROM fact_nav
GROUP BY year, month
ORDER BY year, month;


-- Query 3: SIP Year-over-Year Growth
WITH yearly_sip AS (
    SELECT
        strftime('%Y', transaction_date) AS year,
        SUM(amount_inr) AS sip_amount
    FROM fact_transactions
    WHERE transaction_type = 'SIP'
    GROUP BY year
)
SELECT
    year,
    ROUND(sip_amount, 2) AS sip_amount,
    ROUND(
        (sip_amount - LAG(sip_amount) OVER (ORDER BY year))
        * 100.0
        / LAG(sip_amount) OVER (ORDER BY year),
        2
    ) AS yoy_growth_pct
FROM yearly_sip
ORDER BY year;


-- Query 4: Transactions by State
SELECT
    state,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount_inr), 2) AS total_transaction_amount
FROM fact_transactions
GROUP BY state
ORDER BY total_transaction_amount DESC;


-- Query 5: Funds with Expense Ratio Below 1%
SELECT
    f.scheme_name,
    f.fund_house,
    p.expense_ratio_pct
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
WHERE p.expense_ratio_pct < 1
ORDER BY p.expense_ratio_pct ASC;


-- Query 6: Transaction Type Analysis
SELECT
    transaction_type,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount_inr), 2) AS total_amount,
    ROUND(AVG(amount_inr), 2) AS average_amount
FROM fact_transactions
GROUP BY transaction_type
ORDER BY total_amount DESC;


-- Query 7: 3-Year Fund Performance vs Benchmark
SELECT
    f.scheme_name,
    f.fund_house,
    p.return_3yr_pct,
    p.benchmark_3yr_pct,
    ROUND(
        p.return_3yr_pct - p.benchmark_3yr_pct,
        2
    ) AS performance_difference
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
WHERE p.return_3yr_pct > p.benchmark_3yr_pct
ORDER BY performance_difference DESC;


-- Query 8: Transactions by City Tier
SELECT
    city_tier,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount_inr), 2) AS total_transaction_amount,
    ROUND(AVG(amount_inr), 2) AS average_transaction_amount
FROM fact_transactions
GROUP BY city_tier
ORDER BY average_transaction_amount DESC;


-- Query 9: Fund Distribution by Risk Grade
SELECT
    risk_grade,
    COUNT(*) AS fund_count,
    ROUND(AVG(return_1yr_pct), 2) AS average_1yr_return,
    ROUND(AVG(return_3yr_pct), 2) AS average_3yr_return
FROM fact_performance
GROUP BY risk_grade
ORDER BY fund_count DESC;


-- Query 10: Top 5 Funds by 1-Year Return
SELECT
    f.scheme_name,
    f.fund_house,
    p.return_1yr_pct,
    p.return_3yr_pct,
    p.risk_grade
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
ORDER BY p.return_1yr_pct DESC
LIMIT 5;
"""

with open(queries_path, "w", encoding="utf-8") as f:
    f.write(queries_sql)

print("queries.sql created successfully!")
print("File exists:", queries_path.exists())
print("File path:", queries_path)

queries.sql created successfully!
File exists: True
File path: ..\sql\queries.sql


In [48]:
data_dictionary_path = project_path / "data_dictionary.md"

data_dictionary = """
# Mutual Fund Analytics — Data Dictionary

## Project Overview

This data dictionary documents the datasets, columns, data types,
business definitions, and source references used in the Mutual Fund
Analytics Capstone Project.

---

## 1. nav_history

**Source:** `nav_history.csv`

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| amfi_code | INTEGER | Unique AMFI scheme identifier | nav_history.csv |
| date | DATE | NAV date | nav_history.csv |
| nav | REAL | Net Asset Value of the mutual fund on the given date | nav_history.csv |

---

## 2. investor_transactions

**Source:** `investor_transactions.csv`

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| investor_id | TEXT | Unique investor identifier | investor_transactions.csv |
| transaction_date | DATE | Date on which the transaction occurred | investor_transactions.csv |
| amfi_code | INTEGER | AMFI scheme identifier associated with the transaction | investor_transactions.csv |
| transaction_type | TEXT | Type of transaction: SIP, Lumpsum, or Redemption | investor_transactions.csv |
| amount_inr | REAL | Transaction amount in Indian Rupees | investor_transactions.csv |
| state | TEXT | State associated with the investor | investor_transactions.csv |
| city | TEXT | City associated with the investor | investor_transactions.csv |
| city_tier | TEXT | Classification of the investor's city by tier | investor_transactions.csv |
| age_group | TEXT | Age-group category of the investor | investor_transactions.csv |
| gender | TEXT | Gender category of the investor | investor_transactions.csv |
| annual_income_lakh | REAL | Investor annual income in lakh INR | investor_transactions.csv |
| payment_mode | TEXT | Mode used to make the transaction | investor_transactions.csv |
| kyc_status | TEXT | KYC verification status of the investor | investor_transactions.csv |

---

## 3. scheme_performance

**Source:** `scheme_performance.csv`

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| amfi_code | INTEGER | Unique AMFI scheme identifier | scheme_performance.csv |
| scheme_name | TEXT | Name of the mutual fund scheme | scheme_performance.csv |
| fund_house | TEXT | Mutual fund company/asset management company | scheme_performance.csv |
| category | TEXT | Broad mutual fund category | scheme_performance.csv |
| plan | TEXT | Fund plan type | scheme_performance.csv |
| return_1yr_pct | REAL | One-year return percentage | scheme_performance.csv |
| return_3yr_pct | REAL | Three-year return percentage | scheme_performance.csv |
| return_5yr_pct | REAL | Five-year return percentage | scheme_performance.csv |
| benchmark_3yr_pct | REAL | Three-year benchmark return percentage | scheme_performance.csv |
| alpha | REAL | Alpha measure of fund performance | scheme_performance.csv |
| beta | REAL | Beta measure of market sensitivity | scheme_performance.csv |
| sharpe_ratio | REAL | Sharpe ratio measuring risk-adjusted return | scheme_performance.csv |
| sortino_ratio | REAL | Sortino ratio measuring downside-risk-adjusted return | scheme_performance.csv |
| std_dev_ann_pct | REAL | Annualized standard deviation percentage | scheme_performance.csv |
| max_drawdown_pct | REAL | Maximum observed drawdown percentage | scheme_performance.csv |
| aum_crore | REAL | Assets under management in crore INR | scheme_performance.csv |
| expense_ratio_pct | REAL | Fund expense ratio percentage | scheme_performance.csv |
| morningstar_rating | INTEGER | Morningstar rating assigned to the scheme | scheme_performance.csv |
| risk_grade | TEXT | Risk classification of the mutual fund scheme | scheme_performance.csv |

---

## 4. fund_master

**Source:** `fund_master.csv`

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| amfi_code | INTEGER | Unique AMFI scheme identifier | fund_master.csv |
| fund_house | TEXT | Mutual fund company/asset management company | fund_master.csv |
| scheme_name | TEXT | Name of the mutual fund scheme | fund_master.csv |
| category | TEXT | Broad mutual fund category | fund_master.csv |
| sub_category | TEXT | More specific fund classification | fund_master.csv |
| plan | TEXT | Fund plan type | fund_master.csv |
| launch_date | DATE | Scheme launch date | fund_master.csv |
| benchmark | TEXT | Benchmark associated with the scheme | fund_master.csv |
| expense_ratio_pct | REAL | Expense ratio percentage | fund_master.csv |
| exit_load_pct | REAL | Exit load percentage | fund_master.csv |
| min_sip_amount | REAL | Minimum SIP investment amount | fund_master.csv |
| min_lumpsum_amount | REAL | Minimum lumpsum investment amount | fund_master.csv |
| fund_manager | TEXT | Fund manager associated with the scheme | fund_master.csv |
| risk_category | TEXT | Risk classification of the scheme | fund_master.csv |
| sebi_category_code | TEXT | SEBI category code | fund_master.csv |

---

## 5. portfolio_holdings

**Source:** `portfolio_holdings.csv`

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| amfi_code | INTEGER | AMFI scheme identifier | portfolio_holdings.csv |
| scheme_name | TEXT | Mutual fund scheme name | portfolio_holdings.csv |
| holding_date | DATE | Portfolio holding date | portfolio_holdings.csv |
| company_name | TEXT | Company/security held by the fund | portfolio_holdings.csv |
| sector | TEXT | Sector of the holding | portfolio_holdings.csv |
| holding_pct | REAL | Percentage of portfolio invested in the holding | portfolio_holdings.csv |
| market_value_crore | REAL | Market value of the holding in crore INR | portfolio_holdings.csv |
| rank | INTEGER | Rank of the holding within the portfolio | portfolio_holdings.csv |

---

## 6. benchmark_indices

**Source:** `benchmark_indices.csv`

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| index_name | TEXT | Name of the benchmark index | benchmark_indices.csv |
| date | DATE | Index observation date | benchmark_indices.csv |
| index_value | REAL | Benchmark index value on the given date | benchmark_indices.csv |

---

## 7. aum_by_fund_house

**Source:** `aum_by_fund_house.csv`

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| date | DATE | Reporting date | aum_by_fund_house.csv |
| fund_house | TEXT | Mutual fund company/asset management company | aum_by_fund_house.csv |
| aum_lakh_crore | REAL | Assets under management in lakh crore INR | aum_by_fund_house.csv |
| aum_crore | REAL | Assets under management in crore INR | aum_by_fund_house.csv |
| num_schemes | INTEGER | Number of schemes managed by the fund house | aum_by_fund_house.csv |

---

## 8. monthly_sip_inflows

**Source:** `monthly_sip_inflows.csv`

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| month | DATE | Month of SIP inflow | monthly_sip_inflows.csv |
| sip_inflow_crore | REAL | SIP inflow amount in crore INR | monthly_sip_inflows.csv |
| sip_accounts_lakh | REAL | Number of SIP accounts in lakh | monthly_sip_inflows.csv |
| average_sip_amount | REAL | Average SIP amount | monthly_sip_inflows.csv |
| yoy_growth_pct | REAL | Year-over-year SIP growth percentage | monthly_sip_inflows.csv |
| contribution_pct | REAL | SIP contribution percentage | monthly_sip_inflows.csv |

---

## 9. category_inflows

**Source:** `category_inflows.csv`

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| month | DATE | Month of reported inflow | category_inflows.csv |
| category | TEXT | Mutual fund category | category_inflows.csv |
| inflow_crore | REAL | Inflow amount in crore INR | category_inflows.csv |

---

## 10. industry_folio_count

**Source:** `industry_folio_count.csv`

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| month | DATE | Reporting month | industry_folio_count.csv |
| category | TEXT | Mutual fund category | industry_folio_count.csv |
| folios_lakh | REAL | Number of folios in lakh | industry_folio_count.csv |
| net_change_lakh | REAL | Net change in folios in lakh | industry_folio_count.csv |
| growth_pct | REAL | Growth percentage | industry_folio_count.csv |
| share_pct | REAL | Category share percentage | industry_folio_count.csv |

---

# SQL Star Schema Reference

The SQLite database contains the following tables:

- `dim_fund` — Fund/scheme master dimension
- `dim_date` — Date dimension
- `fact_nav` — Historical NAV observations
- `fact_transactions` — Investor transactions
- `fact_performance` — Fund performance metrics
- `fact_aum` — Fund-house AUM information

## Primary Relationships

- `dim_fund.amfi_code` → fund identifier
- `dim_date.date` → date identifier
- `fact_nav.amfi_code` → `dim_fund.amfi_code`
- `fact_nav.date` → `dim_date.date`
- `fact_transactions.amfi_code` → `dim_fund.amfi_code`
- `fact_transactions.transaction_date` → `dim_date.date`
- `fact_performance.amfi_code` → `dim_fund.amfi_code`
- `fact_aum.date` → `dim_date.date`

---

## Data Cleaning Reference

The following datasets received detailed cleaning and validation during Day 2:

1. `nav_history.csv`
2. `investor_transactions.csv`
3. `scheme_performance.csv`

Cleaning included date parsing, duplicate checks, numeric validation, transaction-type validation, KYC validation, and expense-ratio validation as applicable.

"""

with open(data_dictionary_path, "w", encoding="utf-8") as f:
    f.write(data_dictionary)

print("data_dictionary.md created successfully!")
print("File exists:", data_dictionary_path.exists())
print("File path:", data_dictionary_path)

data_dictionary.md created successfully!
File exists: True
File path: ..\data_dictionary.md
